# BL importer finalization & debug

## Imports

In [1]:
import os
import json

from impresso_essentials.utils import ALL_MEDIA, PARTNER_TO_MEDIA
from copy import deepcopy
from tqdm import tqdm
import pandas as pd
import bs4
from random import shuffle
from text_preparation.importers.bl.detect import BlIssueDir, dir2issue, detect_issues, select_issues
from text_preparation.importers.bl.omni.classes import BlOmniNewspaperPage, BlOmniNewspaperIssue
from PIL import Image, ImageDraw, ImageFont
from text_preparation.utils import draw_box_on_img, coords_to_xywh, coords_to_xy, rescale_coords
from text_preparation.importers.mets_alto.alto import distill_coordinates
from text_preparation.importers import (
    CONTENTITEM_TYPES,
    CONTENTITEM_TYPE_IMAGE,
    CONTENTITEM_TYPE_ADVERTISEMENT,
)

## OmniPage Format

First, adapt BL_ocr_formats.json file to only keep the Aliases and issues corresponding to OmniPage-NLP format.
That will allow to only detect titles for this format and will already help a lot.
The document could have all the issues or be much much smaller with only Alias > NLP > list of dates for the given format. Or directly have the list of paths to mets files of the correct format.

Then the BL_extended_title_list.csv should also be processed to go from alias > NLP > date range > working title and variant titles so that each issue can have its variant title attached to it.

In [2]:
bl_source_data_dir = "/mnt/project_impresso/original/BL"
bl_w_source_data_dir = "/mnt/impresso_ocr_BL"
bl_sample_dir = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/BL"
bl_formats_filename = "BL_ocr_formats.json"
bl_titles_filename = "BL_extended_title_list.csv"
alias_to_NLP_filename = "BL_alias_to_NLP.json"
bl_titles_out_filename = "BL_all_titles.json"
bl_alias_25_08_formats = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/BL/BL_ocr_formats_bl_alias_2025-08-25.json"
bl_media_list_ext_path = os.path.join(bl_sample_dir, bl_titles_filename)
alias_to_NLP = os.path.join(bl_sample_dir, alias_to_NLP_filename)
bl_format_specific_issues = "BL_{format}_issues.json"
RENAMING_INFO_FILE = "renaming_info.json"
BL_IMG_TYPE = "illustration"
BL_AD_TYPE = "advert"
BL_CAPTION_TYPE = "caption"

ocr_formats = ["OmniPage-NLP", "BL-ALIAS", "Nuance-NLP", "ABBYY-ALIAS", "ABBYY-NLP"]

In [3]:
with open(os.path.join(bl_source_data_dir, bl_formats_filename), "r", encoding='utf-8') as fin:
    bl_ocr_formats = json.load(fin)

print(f"Reading: {os.path.join(bl_source_data_dir, bl_formats_filename)}")

print(f"There are {len(bl_ocr_formats)} aliases in {bl_formats_filename}")

Reading: /mnt/project_impresso/original/BL/BL_ocr_formats.json
There are 368 aliases in BL_ocr_formats.json


##### In the original run of ocr formats identification, there was a small bug for Alias `HPNW`, (BL-alias, but listed as UNKNOWN in the original `bl_ocr_formats.json` file) so it needs to be corrected

In [ ]:
#with open(os.path.join(bl_source_data_dir, bl_formats_filename), "r", encoding='utf-8') as fin:
with open(os.path.join(bl_sample_dir, "BL_ocr_formats_old.json"), "r", encoding='utf-8') as fin:
    bl_ocr_formats = json.load(fin)

print(f"Reading: {os.path.join(bl_source_data_dir, bl_formats_filename)}")

# performing a deep copy of the original dict, in order to 
corrected_bl_ocr_formats = deepcopy(bl_ocr_formats)

print(f"There are {len(bl_ocr_formats)} aliases in {bl_formats_filename}")

In [ ]:
def id_bl_alias_format(alias, ocr_files, ocr_issue_dir, ocr_formats) -> dict[str, list]:
    _, nlp, year, month, day = ocr_issue_dir.split(alias)[-1].split("/")
    if alias == "HPNW":
        # there is a mismatch for this alias
        expected_prefix = f"WO1_IPNW_{year}_{month}_{day}-"
    else:
        expected_prefix = f"WO1_{alias}_{year}_{month}_{day}-"
    matching_files = [f for f in ocr_files if expected_prefix in f and f.endswith(".xml")]

    if matching_files:
        # add the matching files to the BL-ALIAS OCR format if there are any
        ocr_formats["BL-ALIAS"] = matching_files

    return ocr_formats

Re-identify the format for this alias, and ensure no issues/files are left unidentified

In [ ]:
incomplete_issues = []
other_formats_issues = []

for year, issues in bl_ocr_formats['HPNW'].items():
    for issue, formats in issues.items():
        if 'UNKNOWN' in formats:
            new_ocr_formats = id_bl_alias_format('HPNW', formats['UNKNOWN'], issue, {})
            if any(x not in new_ocr_formats["BL-ALIAS"] for x in formats['UNKNOWN']):
                print(f"NOTE - {year}-{issue} - More formats to indentify in addition to BL_ALIAS!!")
                incomplete_issues.append(issue)
            corrected_bl_ocr_formats['HPNW'][year][issue] = new_ocr_formats
        if len(formats) != 1:
            # ensure that any other formats are kept
            print(f"WARNING - {year}-{issue} - other formats than unknown found: {formats}")
            other_formats_issues.append(issue)
            for format, files in formats.items():
                if format != 'UNKNOWN':
                    corrected_bl_ocr_formats['HPNW'][year][issue][format]=files
                

incomplete_issues, other_formats_issues

WARNING - 1873-/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/08 - other formats than unknown found: {'ABBYY-NLP': ['0000072_18730208_mets.xml'], 'UNKNOWN': ['WO1_IPNW_1873_02_08-0002-006.xml', 'WO1_IPNW_1873_02_08-0004-025.xml', 'WO1_IPNW_1873_02_08-0004-026.xml', 'WO1_IPNW_1873_02_08-0004-023.xml', 'WO1_IPNW_1873_02_08-0002.xml', 'WO1_IPNW_1873_02_08-0002-011.xml', 'WO1_IPNW_1873_02_08-0002-008.xml', 'WO1_IPNW_1873_02_08-0001-004.xml', 'WO1_IPNW_1873_02_08-0001.xml', 'WO1_IPNW_1873_02_08-0004-024.xml', 'WO1_IPNW_1873_02_08-0004-020.xml', 'WO1_IPNW_1873_02_08-0004-021.xml', 'WO1_IPNW_1873_02_08-0003-019.xml', 'WO1_IPNW_1873_02_08-0002-007.xml', 'WO1_IPNW_1873_02_08-0002-014.xml', 'WO1_IPNW_1873_02_08-0001-002.xml', 'WO1_IPNW_1873_02_08-0004.xml', 'WO1_IPNW_1873_02_08-0003-018.xml', 'WO1_IPNW_1873_02_08-0002-010.xml', 'WO1_IPNW_1873_02_08-0004-027.xml', 'WO1_IPNW_1873_02_08-0002-013.xml', 'WO1_IPNW_1873_02_08-0003-015.xml', 'WO1_IPNW_1873_02_08-0003-017.xml', 'WO1_IPNW_1873_02_

([],
 ['/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/08',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/15',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/01',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/22',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/17',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/24',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/03',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/10',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/31',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/02',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/23',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/30',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/09',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/16',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/11/08',
  '/mnt/project_impresso/original/BL/HPNW/0000072/

In [37]:
len(other_formats_issues)

51

In [ ]:
corrected_bl_ocr_formats['HPNW']['1873']['/mnt/project_impresso/original/BL/HPNW/0000072/1873/11/08']

Ensure that the new corrected version of `BL_ocr_formats` has exactly the same keys and values for all aliases, except the alias in question.

In [33]:
for alias, formats in bl_ocr_formats.items():
    if corrected_bl_ocr_formats[alias] != formats:
        same_keys = corrected_bl_ocr_formats[alias].keys() == formats.keys()
        same_issues = all(issues.keys()==formats[year].keys() for year, issues in corrected_bl_ocr_formats[alias].items())
        print(f"Different formats for alias: {alias}, same keys: {same_keys}, same issues: {same_issues}")

len(bl_ocr_formats), len(corrected_bl_ocr_formats)

Different formats for alias: HPNW, same keys: True, same issues: True


(368, 368)

Now the new version of bl_ocr_formats can be saved in the two places it exists

In [34]:
print(f"writing the corrected OCR formats file to paths {os.path.join(bl_w_source_data_dir, bl_formats_filename)} and {os.path.join(bl_sample_dir, bl_formats_filename)}")

writing the corrected OCR formats file to paths /mnt/impresso_ocr_BL/BL_ocr_formats.json and /home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/BL/BL_ocr_formats.json


In [35]:
with open(os.path.join(bl_w_source_data_dir, bl_formats_filename), "w", encoding='utf-8') as fout:
    json.dump(corrected_bl_ocr_formats, fout, indent=2)

with open(os.path.join(bl_sample_dir, bl_formats_filename), "w", encoding='utf-8') as fout:
    json.dump(corrected_bl_ocr_formats, fout, indent=2)

In [ ]:
len(bl_ocr_formats['HPNW']), bl_ocr_formats['HPNW']

### 1. Create format-specific json files

In [38]:
for format in ocr_formats:
    filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues.format(format=format))
    print(filepath)

/mnt/project_impresso/original/BL/BL_OmniPage-NLP_issues.json
/mnt/project_impresso/original/BL/BL_BL-ALIAS_issues.json
/mnt/project_impresso/original/BL/BL_Nuance-NLP_issues.json
/mnt/project_impresso/original/BL/BL_ABBYY-ALIAS_issues.json
/mnt/project_impresso/original/BL/BL_ABBYY-NLP_issues.json


In [ ]:
mets_paths_per_format = {f: {} for f in ocr_formats}

for alias_idx, (alias, yearly_formats) in enumerate(corrected_bl_ocr_formats.items(), start=1):
    num_years = len(yearly_formats)
    print(f"Starting to process alias {alias} ({alias_idx}/368) - {num_years} years:")

    for year, issue_formats in tqdm(yearly_formats.items()):

        for issue_dir_path, formats in issue_formats.items():

            # keep track of which format is the first match for the priority list
            first_match = True
            for format in ocr_formats:

                if format in formats:
                    if alias not in mets_paths_per_format[format]:
                        # initialize the format dict for this alias is not already done
                        mets_paths_per_format[format][alias] = {
                            "priority_issues": {},
                            "other_issues_also_in_format": {},
                            "non_public_domain_post_1905": {}
                        }
                        
                    # select the mets filename or the first file (for BL alias case)
                    mets_or_example_file = formats[format][0]

                    # directly exclude any title after 1905
                    if int(year) > 1905:
                        mets_paths_per_format[format][alias]["non_public_domain_post_1905"][issue_dir_path] = mets_or_example_file
                    elif first_match:  
                        # set pairs of (issue_path, mets or example file)
                        mets_paths_per_format[format][alias]["priority_issues"][issue_dir_path] = mets_or_example_file
                        # all other formats for this issue are less of a priority
                        first_match = False
                    else:
                        mets_paths_per_format[format][alias]["other_issues_also_in_format"][issue_dir_path] = mets_or_example_file

    print(f"Saving to files the issues: latest alias {alias} ({alias_idx}/368).")

    for format in ocr_formats:
        filepath = os.path.join(bl_w_source_data_dir, bl_format_specific_issues.format(format=format))

        with open(filepath, "w", encoding="utf-8") as fout:
            json.dump(mets_paths_per_format[format], fout, indent=2)

Correcting accordingly the issue lists

In [42]:
bl_format_specific_issues_old = "BL_{format}_issues_old.json"
for format in ocr_formats:
    filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues.format(format=format))
    old_filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues_old.format(format=format))
    print(filepath, old_filepath)

/mnt/project_impresso/original/BL/BL_OmniPage-NLP_issues.json /mnt/project_impresso/original/BL/BL_OmniPage-NLP_issues_old.json
/mnt/project_impresso/original/BL/BL_BL-ALIAS_issues.json /mnt/project_impresso/original/BL/BL_BL-ALIAS_issues_old.json
/mnt/project_impresso/original/BL/BL_Nuance-NLP_issues.json /mnt/project_impresso/original/BL/BL_Nuance-NLP_issues_old.json
/mnt/project_impresso/original/BL/BL_ABBYY-ALIAS_issues.json /mnt/project_impresso/original/BL/BL_ABBYY-ALIAS_issues_old.json
/mnt/project_impresso/original/BL/BL_ABBYY-NLP_issues.json /mnt/project_impresso/original/BL/BL_ABBYY-NLP_issues_old.json


In [43]:
old_issue_lists = {}
new_issue_lists = {}
for format in ocr_formats:
    filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues.format(format=format))
    old_filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues_old.format(format=format))

    with open(filepath, "r", encoding="utf-8") as fin:
        new_issue_lists[format] = json.load(fin)
    with open(old_filepath, "r", encoding="utf-8") as fin:
        old_issue_lists[format] = json.load(fin)

    print(f"{format}: new_issue_lists==old_issue_lists: {new_issue_lists[format]==old_issue_lists[format]}")

OmniPage-NLP: new_issue_lists==old_issue_lists: True
BL-ALIAS: new_issue_lists==old_issue_lists: False
Nuance-NLP: new_issue_lists==old_issue_lists: True
ABBYY-ALIAS: new_issue_lists==old_issue_lists: True
ABBYY-NLP: new_issue_lists==old_issue_lists: False


### 2. Creating a file with all the variant titles

In [4]:
bl_media_lst_ext_raw_df = pd.read_csv(bl_media_list_ext_path, header=0, index_col=0)
print(bl_media_lst_ext_raw_df.info())
bl_media_lst_ext_raw_df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 647 entries, 1 to 628
Data columns (total 12 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Normalized Working Title           647 non-null    object 
 1   Working title (BL)                 647 non-null    object 
 2   Variant Title                      647 non-null    object 
 3   NLP                                647 non-null    int64  
 4   Alias (in file-syst or generated)  647 non-null    object 
 5   Country                            549 non-null    object 
 6   Start Year                         544 non-null    float64
 7   End Year                           544 non-null    float64
 8   Copy already shared with Impresso  647 non-null    object 
 9   Start year in Impresso local copy  647 non-null    int64  
 10  End year in Impresso local copy    647 non-null    int64  
 11  Notes about local copy             87 non-null     object 
dtyp

,Normalized Working Title,Working title (BL),Variant Title,NLP,Alias (in file-syst or generated),Country,Start Year,End Year,Copy already shared with Impresso,Start year in Impresso local copy,End year in Impresso local copy,Notes about local copy
1,Aberdeen Press and Journal,Aberdeen Press and Journal,Aberdeen Journal and General Advertiser,31,ANJO,Scotland,1798.0,1876.0,"Yes, fully",1789,1876,NaN
2,Aberdeen Press and Journal,Aberdeen Press and Journal,Aberdeen Weekly Journal and General Advertiser,32,ANJO,Scotland,1876.0,1900.0,"Yes, fully",1877,1900,There were some small problems in the filenami...
444,Alston Herald and East Cumberland Advertiser,Alston Herald and East Cumberland Advertiser,"Alston Herald, and East Cumberland Advertiser.",3043,AHEC,England,1875.0,1879.0,"Yes, fully",1875,1879,Not separated in the data
492,Alston Herald and East Cumberland Advertiser,Alston Herald and East Cumberland Advertiser,"Alston Herald, and East Cumberland Advertiser",3043,AHEC,England,1880.0,1880.0,"Yes, fully",1880,1880,Not separated in the data
369,Baldwin's London Weekly Journal,Baldwin's London Weekly Journal,"Baldwin's London Weekly Journal, etc",2243,BLWJ,England,1803.0,1836.0,"Yes, fully",1803,1836,NaN


In [5]:
cols_to_keep = ['Alias (in file-syst or generated)','Normalized Working Title', 'Working title (BL)', 'Variant Title', 'NLP', 'Time Period']

bl_media_lst_ext_raw_df['Time Period'] = bl_media_lst_ext_raw_df[['Start year in Impresso local copy', 'End year in Impresso local copy']].apply(lambda x: f"{x[0]}-{x[1]}", axis=1)
reduced_title_df = bl_media_lst_ext_raw_df[cols_to_keep]
reduced_title_df['NLP'] = reduced_title_df['NLP'].apply(lambda x: str(x).zfill(7))
reduced_title_df['Alias-NLP'] = reduced_title_df[['Alias (in file-syst or generated)', 'NLP']].apply(lambda x: f"{x[0]}-{x[1]}", axis=1)

for col in ['Normalized Working Title', 'Working title (BL)', "Variant Title"]:
    reduced_title_df[col] = reduced_title_df[col].apply(
        lambda x: x.rstrip('. ') if isinstance(x, str) else [y.rstrip('. ') for y in x]
    )

reduced_title_df

/scratch/piconti/impresso/dask_tmp/ipykernel_2363885/531057205.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  bl_media_lst_ext_raw_df['Time Period'] = bl_media_lst_ext_raw_df[['Start year in Impresso local copy', 'End year in Impresso local copy']].apply(lambda x: f"{x[0]}-{x[1]}", axis=1)
/scratch/piconti/impresso/dask_tmp/ipykernel_2363885/531057205.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reduced_title_df['NLP'] = reduced_title_df['NLP'].apply(lambda x: str(x).zfill(7))
/scratch/piconti/impresso/dask_tmp/ipykernel_2363885/531057205.py:6: Futur

,Alias (in file-syst or generated),Normalized Working Title,Working title (BL),Variant Title,NLP,Time Period,Alias-NLP
1,ANJO,Aberdeen Press and Journal,Aberdeen Press and Journal,Aberdeen Journal and General Advertiser,0000031,1789-1876,ANJO-0000031
2,ANJO,Aberdeen Press and Journal,Aberdeen Press and Journal,Aberdeen Weekly Journal and General Advertiser,0000032,1877-1900,ANJO-0000032
444,AHEC,Alston Herald and East Cumberland Advertiser,Alston Herald and East Cumberland Advertiser,"Alston Herald, and East Cumberland Advertiser",0003043,1875-1879,AHEC-0003043
492,AHEC,Alston Herald and East Cumberland Advertiser,Alston Herald and East Cumberland Advertiser,"Alston Herald, and East Cumberland Advertiser",0003043,1880-1880,AHEC-0003043
369,BLWJ,Baldwin's London Weekly Journal,Baldwin's London Weekly Journal,"Baldwin's London Weekly Journal, etc",0002243,1803-1836,BLWJ-0002243
...,...,...,...,...,...,...,...
54,WRWA,Wrexham Advertiser,Wrexham Advertiser,Wrexham Weekly Advertiser,0000185,1854-1857,WRWA-0000185
615,WRWA,Wrexham Advertiser,Wrexham Advertiser,Wrexham Advertiser,0000496,1858-1900,WRWA-0000496
55,GNDL,Y Genedl Gymreig,Y Genedl Gymreig,Y Genedl Gymreig,0000059,1877-1900,GNDL-0000059
56,GLAD,Y Goleuad,Y Goleuad,Y Goleuad,0000058,1869-1900,GLAD-0000058


In [6]:
reduced_title_df[reduced_title_df["Alias (in file-syst or generated)"]=="AHEC"]

,Alias (in file-syst or generated),Normalized Working Title,Working title (BL),Variant Title,NLP,Time Period,Alias-NLP
444,AHEC,Alston Herald and East Cumberland Advertiser,Alston Herald and East Cumberland Advertiser,"Alston Herald, and East Cumberland Advertiser",0003043,1875-1879,AHEC-0003043
492,AHEC,Alston Herald and East Cumberland Advertiser,Alston Herald and East Cumberland Advertiser,"Alston Herald, and East Cumberland Advertiser",0003043,1880-1880,AHEC-0003043


In [7]:
bl_titles = reduced_title_df.groupby(['Alias-NLP', 'Time Period']).agg({
        'Normalized Working Title': lambda x: x.unique()[0] if len(x.unique())==1 else x.unique(),
        'Working title (BL)': lambda x: x.unique()[0] if len(x.unique())==1 else x.unique(),
        "Variant Title": lambda x: x.unique()[0] if len(x.unique())==1 else x.unique(),
    },
).reset_index()

bl_titles

,Alias-NLP,Time Period,Normalized Working Title,Working title (BL),Variant Title
0,AATA-0003031,1846-1846,The Agricultural Advertiser and Tenant-Farmers...,Agricultural Advertiser and Tenant-Farmers' Ad...,The Agricultural Advertiser and Tenant-Farmers...
1,AGE52-0003023,1852-1853,The Age 1852,Age 1852,The Age
2,AGMO-0002364,1811-1811,The Anti-Gallican Monitor,Anti-Gallican Monitor,The Anti-Gallican Monitor
3,AGMO-0002365,1811-1814,The Anti-Gallican Monitor,Anti-Gallican Monitor,"The Anti-Gallican Monitor, and Anti-Corsican C..."
4,AGMO-0002366,1815-1817,The Anti-Gallican Monitor,Anti-Gallican Monitor,The Anti-Gallican Monitor
...,...,...,...,...,...
638,YOHD-0000497,1812-1813,The York Herald,York Herald,"The York Herald, County and General Advertiser"
639,YOHD-0000498,1814-1854,The York Herald,York Herald,The York Herald and General Advertiser
640,YOHD-0000499,1855-1889,The York Herald,York Herald,The York Herald
641,YOHD-0000500,1890-1900,The York Herald,York Herald,The Yorkshire Herald and the York Herald


In [8]:
titles_as_dict = bl_titles.to_dict(orient='records')
titles_as_dict

[{'Alias-NLP': 'AATA-0003031',
  'Time Period': '1846-1846',
  'Normalized Working Title': "The Agricultural Advertiser and Tenant-Farmers' Advocate",
  'Working title (BL)': "Agricultural Advertiser and Tenant-Farmers' Advocate",
  'Variant Title': "The Agricultural Advertiser and Tenant-Farmers' Advocate"},
 {'Alias-NLP': 'AGE52-0003023',
  'Time Period': '1852-1853',
  'Normalized Working Title': 'The Age 1852',
  'Working title (BL)': 'Age 1852',
  'Variant Title': 'The Age'},
 {'Alias-NLP': 'AGMO-0002364',
  'Time Period': '1811-1811',
  'Normalized Working Title': 'The Anti-Gallican Monitor',
  'Working title (BL)': 'Anti-Gallican Monitor',
  'Variant Title': 'The Anti-Gallican Monitor'},
 {'Alias-NLP': 'AGMO-0002365',
  'Time Period': '1811-1814',
  'Normalized Working Title': 'The Anti-Gallican Monitor',
  'Working title (BL)': 'Anti-Gallican Monitor',
  'Variant Title': 'The Anti-Gallican Monitor, and Anti-Corsican Chronicle'},
 {'Alias-NLP': 'AGMO-0002366',
  'Time Period': '

In [10]:
bl_titles_json = {}
for record in titles_as_dict:
    if (
        not isinstance(record['Normalized Working Title'], str) or 
        not isinstance(record['Working title (BL)'], str) or 
        not isinstance(record['Variant Title'], str)
    ):
        print(f"More than one value! {record}")
        if record['Alias-NLP']=='BRLU-0002378' and record['Time Period']=='1820-1821':
            record['Variant Title']=record['Variant Title'][0]
        if (
            (record['Alias-NLP']=='DCWR-0003408' and record['Time Period']=='1869-1895') or 
            (record['Alias-NLP']=='MEXA-0003398' and record['Time Period']=='1846-1848')
        ):
            record['Normalized Working Title']=record['Normalized Working Title'][1]
            record['Variant Title']=record['Variant Title'][1]
            record['Working title (BL)']=record['Working title (BL)'][1]
    if record['Alias-NLP'] in bl_titles_json:
        bl_titles_json[record['Alias-NLP']][record['Time Period']] = record
    else:
        bl_titles_json[record['Alias-NLP']] = {record['Time Period']: record}
    """bl_titles_json.update({
       record['Alias-NLP'] : {record['Time Period']: record}
    })"""

with open(os.path.join(bl_w_source_data_dir, bl_titles_out_filename), "w", encoding='utf-8') as fout:
    json.dump(bl_titles_json, fout, indent=2)